# 03 Build Gold Current Tables

Build Power BI-ready current tables from latest staging metadata and manual dictionary tables.

In [ ]:
from pyspark.sql import functions as F
CATALOG_SCHEMA = 'governance'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG_SCHEMA}')

latest = (spark.table(f'{CATALOG_SCHEMA}.stg_table_metadata')
    .groupBy('sync_run_id')
    .agg(F.max('scanned_at').alias('latest_scanned_at'))
    .orderBy(F.desc('latest_scanned_at'))
    .first())
if latest is None:
    raise ValueError('No staging metadata found')
sync_run_id = latest['sync_run_id']
print(f'Latest sync_run_id = {sync_run_id}')


In [ ]:
stg_t = spark.table(f'{CATALOG_SCHEMA}.stg_table_metadata').where(F.col('sync_run_id') == sync_run_id).where(F.col('scan_status') == 'Success')
stg_c = spark.table(f'{CATALOG_SCHEMA}.stg_column_metadata').where(F.col('sync_run_id') == sync_run_id).where(F.col('scan_status') == 'Success')
manual_t = spark.table(f'{CATALOG_SCHEMA}.manual_table_definition')
manual_c = spark.table(f'{CATALOG_SCHEMA}.manual_column_definition')

table_keys = ['workspace_name', 'lakehouse_name', 'schema_name', 'table_name']
column_keys = table_keys + ['column_name']

def table_id_expr():
    return F.sha2(F.lower(F.concat_ws('|', *[F.coalesce(F.col(c).cast('string'), F.lit('')) for c in table_keys])), 256)

def column_id_expr():
    return F.sha2(F.lower(F.concat_ws('|', *[F.coalesce(F.col(c).cast('string'), F.lit('')) for c in column_keys])), 256)

auto_t = (stg_t
    .withColumn('data_object_id', table_id_expr())
    .withColumn('full_table_name', F.concat_ws('.', 'workspace_name', 'lakehouse_name', 'schema_name', 'table_name'))
    .select('data_object_id', 'source_id', 'layer', 'workspace_name', 'lakehouse_name', 'schema_name', 'table_name', 'full_table_name', 'domain', 'owner_team', 'row_count', 'column_count', F.col('scanned_at').alias('last_metadata_sync_at'))
    .dropDuplicates(['data_object_id']))

auto_c = (stg_c
    .withColumn('data_object_id', table_id_expr())
    .withColumn('column_id', column_id_expr())
    .select('column_id', 'data_object_id', 'source_id', 'layer', 'workspace_name', 'lakehouse_name', 'schema_name', 'table_name', 'column_name', 'ordinal_position', 'data_type', 'is_nullable', F.col('scanned_at').alias('last_metadata_sync_at'))
    .dropDuplicates(['column_id']))


In [ ]:
dim_t = (auto_t.alias('a')
    .join(manual_t.alias('m'), on=table_keys, how='left')
    .select('a.data_object_id', 'a.source_id', 'a.layer', 'a.workspace_name', 'a.lakehouse_name', 'a.schema_name', 'a.table_name', 'a.full_table_name', F.coalesce(F.col('m.domain'), F.col('a.domain')).alias('domain'), F.col('m.business_process').alias('business_process'), F.col('m.table_description').alias('table_description'), F.col('m.grain_description').alias('grain_description'), F.col('m.recommended_usage').alias('recommended_usage'), F.coalesce(F.col('m.owner_team'), F.col('a.owner_team')).alias('owner_team'), F.col('m.business_owner').alias('business_owner'), F.col('m.refresh_frequency').alias('refresh_frequency'), F.col('m.refresh_sla').alias('refresh_sla'), F.coalesce(F.col('m.status'), F.lit('Active')).alias('status'), 'a.row_count', 'a.column_count', F.when(F.col('m.table_description').isNull() | (F.trim(F.col('m.table_description')) == ''), 'Missing').otherwise('Complete').alias('table_definition_status'), F.when(F.col('m.grain_description').isNull() | (F.trim(F.col('m.grain_description')) == ''), 'Missing').otherwise('Complete').alias('grain_definition_status'), 'a.last_metadata_sync_at', F.current_timestamp().alias('gold_updated_at')))

manual_c = manual_c.withColumn('is_visible_in_portal_bool', F.when(F.lower(F.trim(F.col('is_visible_in_portal').cast('string'))).isin('false','0','no','n'), F.lit(False)).otherwise(F.lit(True)))
dim_c = (auto_c.alias('a')
    .join(manual_c.alias('m'), on=column_keys, how='left')
    .select('a.column_id', 'a.data_object_id', 'a.source_id', 'a.layer', 'a.workspace_name', 'a.lakehouse_name', 'a.schema_name', 'a.table_name', 'a.column_name', F.coalesce(F.col('m.display_column_name'), F.col('a.column_name')).alias('display_column_name'), 'a.ordinal_position', 'a.data_type', 'a.is_nullable', F.col('m.business_definition').alias('business_definition'), F.col('m.technical_definition').alias('technical_definition'), F.col('m.calculation_logic').alias('calculation_logic'), F.col('m.example_value').alias('example_value'), F.col('m.usage_note').alias('usage_note'), F.col('m.source_table').alias('source_table'), F.col('m.source_column').alias('source_column'), F.coalesce(F.col('m.is_visible_in_portal_bool'), F.lit(True)).alias('is_visible_in_portal'), F.when(F.col('m.business_definition').isNull() | (F.trim(F.col('m.business_definition')) == ''), 'Missing').otherwise('Complete').alias('column_definition_status'), 'a.last_metadata_sync_at', F.current_timestamp().alias('gold_updated_at')))

dim_t.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{CATALOG_SCHEMA}.dim_data_object')
dim_c.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{CATALOG_SCHEMA}.dim_column')


In [ ]:
search = (dim_c.alias('c')
    .join(dim_t.alias('t'), on='data_object_id', how='left')
    .where(F.col('c.is_visible_in_portal') == True)
    .withColumn('search_text', F.lower(F.concat_ws(' ', 't.workspace_name', 't.lakehouse_name', 't.schema_name', 't.table_name', 'c.column_name', 'c.display_column_name', 't.domain', 't.table_description', 't.grain_description', 'c.business_definition', 'c.usage_note'))))
search.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{CATALOG_SCHEMA}.rpt_data_dictionary_search')

missing_table = dim_t.where(F.col('table_definition_status') == 'Missing').select(F.lit('Missing table description').alias('issue_type'), 'layer', 'workspace_name', 'lakehouse_name', 'schema_name', 'table_name', F.lit(None).cast('string').alias('column_name'), 'owner_team', 'last_metadata_sync_at')
missing_grain = dim_t.where(F.col('grain_definition_status') == 'Missing').select(F.lit('Missing grain description').alias('issue_type'), 'layer', 'workspace_name', 'lakehouse_name', 'schema_name', 'table_name', F.lit(None).cast('string').alias('column_name'), 'owner_team', 'last_metadata_sync_at')
missing_col = dim_c.alias('c').join(dim_t.alias('t'), on='data_object_id', how='left').where(F.col('c.column_definition_status') == 'Missing').select(F.lit('Missing column definition').alias('issue_type'), F.col('t.layer'), F.col('t.workspace_name'), F.col('t.lakehouse_name'), F.col('t.schema_name'), F.col('t.table_name'), F.col('c.column_name'), F.col('t.owner_team'), F.col('t.last_metadata_sync_at'))
missing_table.unionByName(missing_grain).unionByName(missing_col).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{CATALOG_SCHEMA}.rpt_missing_definition')

sync_log = (spark.table(f'{CATALOG_SCHEMA}.stg_table_metadata').where(F.col('sync_run_id') == sync_run_id).groupBy('sync_run_id').agg(F.count('*').alias('scanned_table_items'), F.sum(F.when(F.col('scan_status') == 'Success', 1).otherwise(0)).alias('success_table_items'), F.sum(F.when(F.col('scan_status') == 'Failed', 1).otherwise(0)).alias('failed_table_items'), F.max('scanned_at').alias('sync_finished_at')).withColumn('sync_status', F.when(F.col('failed_table_items') > 0, 'Partial Success').otherwise('Success')).withColumn('created_at', F.current_timestamp()))
sync_log.write.format('delta').mode('append').saveAsTable(f'{CATALOG_SCHEMA}.fact_metadata_sync_run')
